Primer Trabajo FSI: Redes Neuronales

Importamos las librerías

In [2]:
import torch
from torch import nn, optim
from torchvision import datasets, transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import pandas as pd
import numpy as np
import os
import kagglehub
from torchsummary import summary
import torch.nn.functional as F
import csv

C:\Users\Daniel\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Como tenemos una carpeta con las imágenes y otra con las etiquetas (YOLO) lo primero que debemos hacer es modificar este formato para tener un fichero csv con las etiquetas.

In [4]:
# --- Configuración del Dataset de KaggleHub ---
KAGGLE_DATASET_ID = 'pkdarabi/cardetection'

print(f"Descargando dataset: {KAGGLE_DATASET_ID}")
KAGGLE_DOWNLOAD_PATH = kagglehub.dataset_download(KAGGLE_DATASET_ID)
print(f"Dataset descargado en: {KAGGLE_DOWNLOAD_PATH}")

# Extensiones estándar YOLO
EXT_IMAGEN = ".jpg"
EXT_LABEL = ".txt"

base_data_path = os.path.join(KAGGLE_DOWNLOAD_PATH, "car")
DATA_DIR = './'
DIVISIONES = ["train", "valid", "test"]

def procesar_split(split_name):
    print(f"\n=== Procesando split: {split_name.upper()} ===")

    img_dir = os.path.join(KAGGLE_DOWNLOAD_PATH, "car", split_name, "images")
    label_dir = os.path.join(KAGGLE_DOWNLOAD_PATH, "car", split_name, "labels")

    csv_output = f"labels_{split_name}.csv"
    datos = []

    if not os.path.exists(img_dir):
        print(f"ERROR: No existe el directorio {img_dir}")
        return

    for archivo in os.listdir(img_dir):
        if archivo.lower().endswith(EXT_IMAGEN):
            nombre_base = os.path.splitext(archivo)[0]
            ruta_label = os.path.join(label_dir, nombre_base + EXT_LABEL)

            if os.path.exists(ruta_label):
                with open(ruta_label, "r") as f:
                    lineas = f.readlines()

                for linea in lineas:
                    partes = linea.strip().split()

                    if len(partes) == 5:
                        clase = int(partes[0])
                        x_center = float(partes[1])
                        y_center = float(partes[2])
                        width = float(partes[3])
                        height = float(partes[4])

                        datos.append({
                            "nombre_archivo": archivo,
                            "clase_indice": clase,
                            "x_center": x_center,
                            "y_center": y_center,
                            "width": width,
                            "height": height
                        })
                    else:
                        print(f"⚠️ Línea incorrecta en {ruta_label}: {linea.strip()}")

    df = pd.DataFrame(datos)
    df.to_csv(csv_output, index=False)

    print(f"CSV creado: {csv_output}")
    print(f"Cajas encontradas: {len(df)}")
    if not df.empty:
        print(df["clase_indice"].value_counts().sort_index())
    print("----------------------------")

# Procesar train, valid, test
for split in DIVISIONES:
    procesar_split(split)


Descargando dataset: pkdarabi/cardetection
Dataset descargado en: C:\Users\Daniel\.cache\kagglehub\datasets\pkdarabi\cardetection\versions\5

=== Procesando split: TRAIN ===
CSV creado: labels_train.csv
Cajas encontradas: 4298
clase_indice
0     542
1     585
2      19
3     267
4     101
5     252
6     285
7     334
8     235
9     283
10    301
11    318
12    323
13    168
14    285
Name: count, dtype: int64
----------------------------

=== Procesando split: VALID ===
CSV creado: labels_valid.csv
Cajas encontradas: 944
clase_indice
0     122
1     108
3      52
4      17
5      60
6      56
7      74
8      55
9      71
10     76
11     78
12     56
13     38
14     81
Name: count, dtype: int64
----------------------------

=== Procesando split: TEST ===
CSV creado: labels_test.csv
Cajas encontradas: 770
clase_indice
0     110
1      94
2       3
3      46
4      21
5      44
6      46
7      60
8      53
9      50
10     45
11     53
12     61
13     34
14     50
Name: count, dty

In [5]:
# Preparación de los CSVs corregidos
def procesar_csv(archivo_entrada, archivo_salida):
    print(f"\n--- Iniciando procesamiento de: '{archivo_entrada}' ---")
    
    filas_eliminadas = 0
    filas_modificadas = 0
    filas_totales_escritas = 0

    try:
        with open(archivo_entrada, mode='r', newline='') as infile, \
             open(archivo_salida, mode='w', newline='') as outfile:

            reader = csv.reader(infile)
            writer = csv.writer(outfile)

            try:
                header = next(reader)
                writer.writerow(header)
            except StopIteration:
                print(f"Error: El archivo '{archivo_entrada}' está vacío.")
                return

            for row in reader:
                if not row:
                    continue

                try:
                    clase_index = int(row[1])
                    
                    if clase_index == 2:
                        filas_eliminadas += 1
                        continue

                    elif clase_index > 2:
                        clase_index -= 1
                        filas_modificadas += 1
                        row[1] = str(clase_index)
                    
                    writer.writerow(row)
                    filas_totales_escritas += 1

                except ValueError:
                    print(f"Advertencia: Se omitió una fila mal formada (índice no numérico): {row}")
                except IndexError:
                    print(f"Advertencia: Se omitió una fila mal formada (faltan columnas): {row}")

        print(f"Resultados guardados en: '{archivo_salida}'")
        print(f"Filas eliminadas: {filas_eliminadas}")
        print(f"Filas re-mapeadas: {filas_modificadas}")
        print(f"Total de filas escritas: {filas_totales_escritas}")

    except FileNotFoundError:
        print(f"Error: No se encontró el archivo de entrada '{archivo_entrada}'.")
    except Exception as e:
        print(f"Ocurrió un error inesperado con '{archivo_entrada}': {e}")

archivos_a_procesar = [
    ('./labels_train.csv', './train_corregido.csv'),
    ('./labels_valid.csv', './valid_corregido.csv'),
    ('./labels_test.csv', './test_corregido.csv')
]
print("Iniciando el procesamiento de todos los archivos...")

for entrada, salida in archivos_a_procesar:
    procesar_csv(entrada, salida)

print("\n--- ¡Proceso de todos los archivos completado! ---")

Iniciando el procesamiento de todos los archivos...

--- Iniciando procesamiento de: './labels_train.csv' ---
Resultados guardados en: './train_corregido.csv'
Filas eliminadas: 19
Filas re-mapeadas: 3152
Total de filas escritas: 4279

--- Iniciando procesamiento de: './labels_valid.csv' ---
Resultados guardados en: './valid_corregido.csv'
Filas eliminadas: 0
Filas re-mapeadas: 714
Total de filas escritas: 944

--- Iniciando procesamiento de: './labels_test.csv' ---
Resultados guardados en: './test_corregido.csv'
Filas eliminadas: 3
Filas re-mapeadas: 563
Total de filas escritas: 767

--- ¡Proceso de todos los archivos completado! ---


Ahora ya podemos crear una instancia de la clase YOLODataset

In [6]:
# --- FRAGMENTO NÚMERO 2 ---
print("\n--- Cargando Datasets y DataLoaders desde los CSVs generados ---")

datasets = {}
dataloaders = {}

IMAGE_SIZE = (416, 416)
BATCH_SIZE = 32

# Normalización OBLIGATORIA para modelos pre-entrenados
normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                 std=[0.229, 0.224, 0.225])

# 1. Transformaciones para ENTRENAMIENTO (con aumento de datos)
train_transform = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.RandomHorizontalFlip(p=0.5), # Volteo aleatorio
    transforms.RandomRotation(15),           # Rotación aleatoria
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2), # Cambios de color
    transforms.ToTensor(),                   # Convertir a Tensor
    normalize,                               # Normalizar
])

# 2. Transformaciones para VALIDACIÓN y TEST (solo limpiar)
val_transform = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.ToTensor(),
    normalize,
])

# Esta clase ahora funcionará porque leerá los CSVs que acabamos de crear.
class DataSet(Dataset):
    def __init__(self, archivo_csv, directorio_imagenes, transform=None):
        self.directorio_imagenes = directorio_imagenes
        self.transform = transform
        
        try:
            self.full_labels_df = pd.read_csv(archivo_csv)
        except FileNotFoundError:
            print(f"  - ¡Error! No se encontró el archivo CSV: {archivo_csv}")
            self.full_labels_df = pd.DataFrame(columns=['nombre_archivo'])
            
        self.imagenes_unicas = self.full_labels_df['nombre_archivo'].unique()
        self.labels_grouped = self.full_labels_df.groupby('nombre_archivo')

    def __len__(self):
        return len(self.imagenes_unicas)

    def __getitem__(self, idx):
        image_name = self.imagenes_unicas[idx]
        image_path = os.path.join(self.directorio_imagenes, image_name)
        
        try:
            image = Image.open(image_path).convert('RGB')
        except FileNotFoundError:
            return torch.zeros(3, IMAGE_SIZE[0], IMAGE_SIZE[1]), torch.tensor(-1) 

        boxes_df = self.labels_grouped.get_group(image_name)
        clase_idx = int(boxes_df['clase_indice'].iloc[0])
        label = torch.tensor(clase_idx, dtype=torch.long)

        if self.transform:
            image = self.transform(image)

        return image, label
 
for label in DIVISIONES:
    # Construir rutas
    ruta_csv = os.path.join(DATA_DIR, f"CSVs/{label}_corregido.csv") 
    ruta_imgs = os.path.join(base_data_path, label, "images")
    
    print(f"\nProcesando conjunto de datos: {label}")
    print(f"Ruta del CSV: {ruta_csv}")
    print(f"Ruta de las imágenes: {ruta_imgs}")

    if not os.path.exists(ruta_csv) or not os.path.exists(ruta_imgs):
        print("  - Error: No se encontró el CSV o el directorio de imágenes.")

    else:
        # Asignar la transformación correcta según el split
        if label == 'train':
            transform_actual = train_transform
        else:
            transform_actual = val_transform
            
        # Crear el Dataset
        dataset = DataSet(archivo_csv=ruta_csv, 
                          directorio_imagenes=ruta_imgs, 
                          transform=transform_actual)
        
        if len(dataset) > 0:
            print(f"  - Tipo de objeto creado: {type(dataset)}")
            print(f"  - Número total de imágenes: {len(dataset)}")

            datasets[label] = dataset
            
            # --- OPTIMIZACIÓN DE GPU ---
            dataloaders[label] = DataLoader(
                dataset,
                batch_size=BATCH_SIZE,
                shuffle=(label == 'train'),
                pin_memory=True,            # Acelera la transferencia a GPU
            )
            print(f"  - DataLoader para '{label}' creado (OPTIMIZADO).")

        else:
            print("  - Error: El dataset está vacío (CSV vacío o no se pudo leer).")

print("\n--- Proceso de carga completado ---")


--- Cargando Datasets y DataLoaders desde los CSVs generados ---

Procesando conjunto de datos: train
Ruta del CSV: ./CSVs/train_corregido.csv
Ruta de las imágenes: C:\Users\Daniel\.cache\kagglehub\datasets\pkdarabi\cardetection\versions\5\car\train\images
  - Error: No se encontró el CSV o el directorio de imágenes.

Procesando conjunto de datos: valid
Ruta del CSV: ./CSVs/valid_corregido.csv
Ruta de las imágenes: C:\Users\Daniel\.cache\kagglehub\datasets\pkdarabi\cardetection\versions\5\car\valid\images
  - Error: No se encontró el CSV o el directorio de imágenes.

Procesando conjunto de datos: test
Ruta del CSV: ./CSVs/test_corregido.csv
Ruta de las imágenes: C:\Users\Daniel\.cache\kagglehub\datasets\pkdarabi\cardetection\versions\5\car\test\images
  - Error: No se encontró el CSV o el directorio de imágenes.

--- Proceso de carga completado ---


A continuación creamos la red neuronal que entrenaremos con el dataset que hemos creado.

In [ ]:
class SimpleNN(nn.Module):
    def __init__(self):
        super(SimpleNN, self).__init__()
        self.fc1 = nn.Linear(3*416*416, 512)
        self.fc2 = nn.Linear(512, 128)    # Capa oculta con 128 neuronas
        self.fc3 = nn.Linear(128, 14)      # Capa de salida con 14 clases (0-13)
        self.activation = nn.Sigmoid()        # Función de activación Sigmoide
        self.softmax = nn.Softmax(dim=1)  # Función softmax para la capa de salida

    def forward(self, x):
        # x = x.view(-1, 416*416)            
        x = x.view(x.size(0), -1)       # Aplanar la imagen de 416x416 a un vector de 173056
        #print(x.shape)                  # Mostrar la forma del tensor después de aplanarlo
        x = self.fc1(x)                
        x = self.activation(x)            # Función de activación Sigmoide en la capa oculta
        #print(x.shape)                   # Mostrar la forma del tensor después de la primera capa
        x = self.fc2(x)                  # Capa de salida
        x = self.activation(x)
        x = self.fc3(x)
        #print(x.shape)                   # Mostrar la forma del tensor después de la segunda capa
        x = self.softmax(x)              # Aplicar softmax para obtener probabilidades
        return x

Definimos la función de perdida y el optimizador.

In [8]:
model = SimpleNN()
summary(model, (3, 416, 416)) # Resumen del modelo
# El modelo y las dimension de entrad de los datos


----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Linear-1                  [-1, 512]     265,814,528
           Sigmoid-2                  [-1, 512]               0
            Linear-3                  [-1, 128]          65,664
           Sigmoid-4                  [-1, 128]               0
            Linear-5                   [-1, 15]           1,935
           Softmax-6                   [-1, 15]               0
Total params: 265,882,127
Trainable params: 265,882,127
Non-trainable params: 0
----------------------------------------------------------------
Input size (MB): 1.98
Forward/backward pass size (MB): 0.01
Params size (MB): 1014.26
Estimated Total Size (MB): 1016.25
----------------------------------------------------------------


Ahora ya podemos entrenar la red.

In [23]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = SimpleNN().to(DEVICE)
criterion = nn.MSELoss()
optimizer = optim.SGD(model.parameters(), lr=0.5)

EPOCHS = 10
for epoch in range(EPOCHS):
    running_loss = 0.0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(inputs)
        labels_one_hot = F.one_hot(labels, num_classes=15).float()  # 15 clases
        loss = criterion(outputs, labels_one_hot)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    print(f"[Epoch {epoch + 1}] loss: {running_loss / len(train_loader):.3f}")


[Epoch 1] loss: 0.062
[Epoch 2] loss: 0.061
[Epoch 3] loss: 0.061
[Epoch 4] loss: 0.060
[Epoch 5] loss: 0.060
[Epoch 6] loss: 0.058
[Epoch 7] loss: 0.057
[Epoch 8] loss: 0.056
[Epoch 9] loss: 0.055
[Epoch 10] loss: 0.054


Y por último evaluamos la red.

In [33]:
# --- Configuración de rutas para test ---
IMAGEN_DIR_TEST = os.path.join(KAGGLE_DOWNLOAD_PATH, 'car', 'test', 'images') 
ETIQUETAS_DIR_TEST = os.path.join(KAGGLE_DOWNLOAD_PATH, 'car', 'test', 'labels')
CSV_SALIDA_TEST = 'yolo_labels_test.csv'

datos = []

print(f"\nEscaneando imágenes en: {IMAGEN_DIR_TEST}")

# 2. Iterar sobre los archivos de imagen para emparejar
if not os.path.exists(IMAGEN_DIR_TEST):
    print(f"ERROR: No se encontró el directorio de imágenes en {IMAGEN_DIR_TEST}. Revisa la estructura del dataset.")
else:
    for archivo_imagen in os.listdir(IMAGEN_DIR_TEST):
        if archivo_imagen.lower().endswith(EXTENSION_IMAGEN):
            nombre_base = os.path.splitext(archivo_imagen)[0]
            ruta_etiqueta = os.path.join(ETIQUETAS_DIR_TEST, nombre_base + EXTENSION_ETIQUETA)
            
            if os.path.exists(ruta_etiqueta):
                with open(ruta_etiqueta, 'r') as f:
                    lineas = f.readlines()
                
                for linea in lineas:
                    partes = linea.strip().split()
                    if len(partes) == 5:
                        clase_idx = int(partes[0])
                        x_center = float(partes[1])
                        y_center = float(partes[2])
                        width = float(partes[3])
                        height = float(partes[4])
                        datos.append({
                            'nombre_archivo': archivo_imagen,
                            'clase_indice': clase_idx,
                            'x_center': x_center,
                            'y_center': y_center,
                            'width': width,
                            'height': height
                        })
                    else:
                        print(f"¡Advertencia! Línea con formato incorrecto en {ruta_etiqueta}: {linea.strip()}")
            # No se requiere else: es normal que algunas imágenes no tengan etiquetas

# 3. Crear DataFrame y guardar CSV
df_test = pd.DataFrame(datos)
df_test.to_csv(CSV_SALIDA_TEST, index=False)

print("\n--- Resumen Test ---")
print(f"Total de cajas delimitadoras encontradas: {len(df_test)}")
if not df_test.empty:
    print("Conteo de objetos por clase (Índice):")
    print(df_test['clase_indice'].value_counts().sort_index())
print(f"¡CSV de test creado exitosamente en: {CSV_SALIDA_TEST}!")

# 4. Definir las rutas usando la ruta de descarga de Kaggle
ruta_csv_test = os.path.join(DATA_DIR, CSV_SALIDA_TEST) # El CSV se creó en el directorio actual
# IMPORTANTE: Definir la ruta de imágenes APUNTANDO al subdirectorio 'train/images'
ruta_imgs_test = os.path.join(KAGGLE_DOWNLOAD_PATH, 'car', 'test', 'images') 

# Crear DataLoader de test
dataset_yolo_test = YOLODataset(archivo_csv=ruta_csv_test, directorio_imagenes=ruta_imgs_test, transform=transform)
test_loader = DataLoader(dataset_yolo_test, batch_size=BATCH_SIZE, shuffle=False)

print(f"\nTest DataLoader creado con {len(test_loader)} batches")



Escaneando imágenes en: C:\Users\Daniel\.cache\kagglehub\datasets\pkdarabi\cardetection\versions\5\car\test\images

--- Resumen Test ---
Total de cajas delimitadoras encontradas: 770
Conteo de objetos por clase (Índice):
clase_indice
0     110
1      94
2       3
3      46
4      21
5      44
6      46
7      60
8      53
9      50
10     45
11     53
12     61
13     34
14     50
Name: count, dtype: int64
¡CSV de test creado exitosamente en: yolo_labels_test.csv!

Test DataLoader creado con 40 batches


In [34]:
# Evaluación de la red
def evaluate(model, test_loader):
    model.eval()  # Poner el modelo en modo evaluación
    correct = 0
    total = test_loader.dataset.__len__()  # Total de muestras en el conjunto de test
    print(f'Total de muestras en el conjunto de test: {total}')
    with torch.no_grad():  # No calcular gradientes
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)  # Mover datos al dispositivo
            outputs = model(inputs)  # Forward pass
            _, predicted = torch.max(outputs.data, 1)  # Obtener las predicciones
            correct += (predicted == labels).sum().item()  # Actualizar el contador de aciertos
    accuracy = 100 * correct / total if total > 0 else 0
    print(f'Accuracy: {accuracy:.2f}%')

In [35]:
evaluate(model, test_loader)

Total de muestras en el conjunto de test: 637
Accuracy: 29.67%
